# 함수와 합성함수 — 딥러닝은 함수를 수백 겹 쌓은 것

**Zero to 머신러닝 딥러닝 Master — Medium 시리즈 #21** · 원문: 책 01-08 「함수와 합성함수 — 딥러닝의 구조」

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/otoloism/Zero_to_ML_Master/blob/main/medium-series/21-functions-composition/21_functions_composition.ipynb)

합성함수 g(f(x))와 f(g(x))가 다르다는 것, 신경망이 층(함수)을 겹겹이 합성한 것이라는 것, 활성화 함수 없이 쌓은 선형 층은 행렬 하나로 붕괴한다는 것을 확인하고, sigmoid·ReLU·tanh, 연쇄법칙(수치미분 검산), 기울기 소실(0.25ⁿ)과 ReLU의 해법까지 코드로 따라갑니다.

- 📝 Medium 글: **TODO: Medium 게시 후 URL 입력** (`https://medium.com/p/...`)
- 📘 책 원문: [01-08 함수와 합성함수 — 딥러닝의 구조](https://wikidocs.net/439745)
- 🏠 [mldict.net](https://mldict.net) · ✍️ [기술 블로그](https://wikidocs.net/blog/@mldict/) · 📚 [책 전체 목차](https://wikidocs.net/book/21464)

## 1. 1단원 — 합성함수와 순서의 중요성

f(x)=2x+1, g(x)=x² 로 g(f(3)) = 49, f(g(3)) = 19 를 계산해 합성 순서가 결과를 바꾼다는 것을 확인합니다.

In [1]:
# 함수 = 입력을 규칙대로 바꿔 출력하는 상자
def f(x):
    return 2 * x + 1        # 규칙: 2배 하고 1 더하기

def g(x):
    return x ** 2          # 규칙: 제곱하기

# 합성함수 g(f(x)) — 먼저 f, 그 결과를 g에 넣기
x = 3
print("f(3)     =", f(3))              # 3 → 7
print("g(f(3))  =", g(f(3)))           # 7 → 49
print("f(g(3))  =", f(g(3)))           # 순서를 바꾸면? 9 → 19
print("순서가 중요한가?", g(f(3)) != f(g(3)))

f(3)     = 7
g(f(3))  = 49
f(g(3))  = 19
순서가 중요한가? True


## 2. 1단원 — 층을 함수로 이어 붙이기

한 층 = 선형변환 + 활성화 함수. layer2(layer1(x)) 가 곧 합성함수입니다.

In [2]:
import numpy as np

# 신경망 한 층 = 행렬곱 + 활성화 함수 (하나의 함수)
def layer1(x):
    return np.maximum(0, 2 * x - 1)    # 선형변환 후 ReLU

def layer2(x):
    return 3 * x + 0.5

# 3층짜리 신경망 = layer2(layer1(입력))처럼 함수를 겹겹이
x = 1.0
h1 = layer1(x)            # 1층 통과
y  = layer2(h1)           # 2층 통과
print("입력 x =", x)
print("1층 출력 =", h1)
print("최종 출력 =", y)
print("\n신경망은 결국 함수를 여러 겹 합성한 것: y = layer2(layer1(x))")

입력 x = 1.0
1층 출력 = 1.0
최종 출력 = 3.5

신경망은 결국 함수를 여러 겹 합성한 것: y = layer2(layer1(x))


## 3. 2단원 — 선형 두 층 = 선형 한 층

활성화 함수 없이 A, B 두 선형 층을 합치면 A @ B 라는 행렬 하나가 됩니다.

In [3]:
import numpy as np

# 활성화 함수 없이 선형변환만 쌓으면?
A = np.array([[2, 0], [0, 3]])     # 1층 (선형)
B = np.array([[1, 1], [0, 1]])     # 2층 (선형)

합쳐진_변환 = A @ B                  # 두 층을 합치면...
print("A @ B =\n", 합쳐진_변환)
print("→ 여전히 '행렬 하나' = 선형변환 하나")
print("층을 아무리 쌓아도 직선밖에 못 그린다!")

A @ B =
 [[2 2]
 [0 3]]
→ 여전히 '행렬 하나' = 선형변환 하나
층을 아무리 쌓아도 직선밖에 못 그린다!


## 4. 2단원 — sigmoid, ReLU, tanh

세 활성화 함수에 같은 입력을 넣어 0~1, 0 이상, -1~1 로 눌러 담기는 모습을 비교합니다.

In [4]:
import numpy as np

z = np.array([-2, -0.5, 0, 0.5, 2])   # 입력값들

sigmoid = 1 / (1 + np.exp(-z))         # 0~1로 눌러 담기
relu    = np.maximum(0, z)             # 음수는 0, 양수는 그대로
tanh    = np.tanh(z)                   # -1~1로 눌러 담기

print("입력 z  :", z)
print("sigmoid :", sigmoid.round(3))
print("relu    :", relu)
print("tanh    :", tanh.round(3))

입력 z  : [-2.  -0.5  0.   0.5  2. ]
sigmoid : [0.119 0.378 0.5   0.622 0.881]
relu    : [0.  0.  0.  0.5 2. ]
tanh    : [-0.964 -0.462  0.     0.462  0.964]


## 5. 3단원 — 연쇄법칙 dy/dx = dy/du × du/dx

y = (2x+1)² 를 u = 2x+1 로 나눠 미분을 곱합니다 (x = 3 에서 28).

In [5]:
# 연쇄법칙:  dy/dx = (dy/du) × (du/dx)
# y = g(f(x)),  f(x)=2x+1,  g(u)=u²
x = 3
u = 2*x + 1                    # f(x)

dg_du = 2 * u                  # g'(u) = 2u
df_dx = 2                      # f'(x) = 2
dy_dx = dg_du * df_dx          # 연쇄법칙으로 곱하기

print("u = f(x) =", u)
print("dy/du =", dg_du, ", du/dx =", df_dx)
print("dy/dx = dy/du × du/dx =", dy_dx)

u = f(x) = 7
dy/du = 14 , du/dx = 2
dy/dx = dy/du × du/dx = 28


## 6. 3단원 — 연쇄법칙 결과를 수치미분으로 검산

미분의 정의 (y(x+h) − y(x)) / h 로 직접 재서 연쇄법칙 결과와 비교합니다.

In [6]:
# 연쇄법칙 결과가 맞는지 '수치미분'으로 검산
def y(x):
    return (2*x + 1) ** 2      # g(f(x))를 통째로

x = 3
h = 1e-6
근사기울기 = (y(x + h) - y(x - h)) / (2 * h)   # 미분의 정의

print("연쇄법칙 결과 :", 28)
print("수치미분 결과 :", round(근사기울기, 4))
print("일치하는가?", abs(근사기울기 - 28) < 0.001)

연쇄법칙 결과 : 28
수치미분 결과 : 28.0
일치하는가? True


## 7. 4단원 — 층이 깊어질수록 기울기가 사라진다

sigmoid 미분의 최댓값 0.25 를 층 수만큼 곱해 기울기 소실을 숫자로 확인합니다.

In [7]:
import numpy as np

# sigmoid의 미분은 최대 0.25 — 층마다 이 값이 곱해진다
최대기울기 = 0.25

print("역전파에서 층을 거슬러 갈수록 기울기가 곱해진다:")
for 층수 in [1, 5, 10, 20]:
    남은기울기 = 최대기울기 ** 층수
    print(f"  {층수:2d}층 통과 → 0.25^{층수} = {남은기울기:.2e}")

print("\n20층이면 기울기가 사실상 0 → 앞쪽 층은 학습이 멈춘다")

역전파에서 층을 거슬러 갈수록 기울기가 곱해진다:
   1층 통과 → 0.25^1 = 2.50e-01
   5층 통과 → 0.25^5 = 9.77e-04
  10층 통과 → 0.25^10 = 9.54e-07
  20층 통과 → 0.25^20 = 9.09e-13

20층이면 기울기가 사실상 0 → 앞쪽 층은 학습이 멈춘다


## 8. 4단원 — ReLU 미분은 곱해도 1

양수 구간에서 ReLU 의 미분은 항상 1 이라 층을 거쳐도 기울기가 줄지 않습니다.

In [8]:
import numpy as np

# ReLU의 미분: 양수 구간에서 항상 1 → 곱해도 안 줄어든다
def relu_미분(z):
    return (z > 0).astype(float)    # z>0이면 1, 아니면 0

z = np.array([-1, 0.5, 2, 3])
print("z         :", z)
print("ReLU 미분 :", relu_미분(z))

기울기 = 1.0
for 층 in range(20):
    기울기 *= 1.0                    # 양수 구간이면 1을 곱함
print("\n20층 통과 후 기울기 =", 기울기, "→ 소실되지 않음!")

z         : [-1.   0.5  2.   3. ]
ReLU 미분 : [0. 1. 1. 1.]

20층 통과 후 기울기 = 1.0 → 소실되지 않음!


## 🔗 함께 보기

- 📘 **책 (WikiDocs)** — [Zero to 머신러닝 딥러닝 Master](https://wikidocs.net/book/21464)
- 🏠 **홈페이지** — [mldict.net](https://mldict.net)
- ✍️ **기술 블로그** — [wikidocs.net/blog/@mldict](https://wikidocs.net/blog/@mldict/)
- 💻 **전체 코드 저장소** — [github.com/otoloism/Zero_to_ML_Master](https://github.com/otoloism/Zero_to_ML_Master)

👉 더 자세한 설명과 그림은 [책 원문](https://wikidocs.net/439745) 에서 확인하세요.